# Milestone 1: data and EDA

This notebook is generated by `tools/build_notebooks.py`. Edit the builder, not this file: a test fails if the two drift apart.

It follows course modules 2 to 4 (get the prices, clean them, take logs, describe them), plus the 90/10 split, which the course makes in module 7. Every code cell calls the package `arima_forecasting`.

It reads the JSON files the exporter writes in `app/public/data/`, which are committed (`prices.json` and `eda.json` too, since the owner's decision of 2026-10-29; see `DATA.md`). The first code cell stops with build instructions if they are missing. The raw snapshot stays outside git; one cell below rebuilds the files from it, if it is on this machine, and checks they match.

The pictures live in the app's price explorer (`cd app && npm run dev`). This notebook prints numbers, so a test can check them.

In [ ]:
from arima_forecasting import config, data, eda, export, prep
from arima_forecasting.artifacts import find_app_data, read_artifact, table_from_prices

app_data = find_app_data()
missing = [n for n in ("prices", "eda") if not (app_data / f"{n}.json").is_file()]
if missing:
    raise SystemExit(
        f"{', '.join(n + '.json' for n in missing)} not in this copy. "
        "Run: uv run python tools/fetch_prices.py, then uv run python tools/export_app_data.py"
    )
prices = read_artifact(app_data, "prices")["data"]
eda_data = read_artifact(app_data, "eda")["data"]
print("read", app_data / "prices.json", "and", app_data / "eda.json")

## Module 2: getting the data once

The course calls `yf.download('AAPL', start='2015-01-01', end='2023-12-31')` and keeps the result in memory. Two things change here.

1. **Every argument is explicit**, all 19 of them. A default can change between yfinance versions, and two of them matter: `auto_adjust=True` makes `Close` the split- and dividend-adjusted close, and `multi_level_index=True` gives two-level columns.
2. **The download is a snapshot.** Yahoo back-adjusts its history after every dividend, so two downloads months apart disagree. We download once, write the file, and record its SHA-256 in a manifest. `prices.json` repeats that SHA-256.

In [ ]:
request = data.DOWNLOAD_REQUEST
for key in ("tickers", "start", "end", "interval", "auto_adjust", "multi_level_index", "keepna"):
    print(f"{key:>17}: {request[key]!r}")
source = prices["source"]
print()
print("snapshot sha256:", source["raw_sha256"])
print("raw rows:       ", source["raw_rows"])
print("downloaded at:  ", source["downloaded_at"], "with yfinance", source["yfinance_version"])

## The MultiIndex trap

yfinance 1.7.0 returns two-level columns, `(Price, Ticker)`, even for one ticker. So `frame["Close"]` is a one-column **DataFrame**, not a Series. Subtract it from a Series and pandas lines the Series' index up with the DataFrame's *columns*: every value becomes NaN. `normalize_download` checks the ticker level and drops it, once, at the edge.

The frame below is built by hand in yfinance's shape. Nothing is downloaded.

In [ ]:
import pandas as pd

days = pd.DatetimeIndex(["2015-01-02", "2015-01-05", "2015-01-06"], name="Date")
columns = pd.MultiIndex.from_tuples([(name, "AAPL") for name in ("Close", "High", "Low", "Open", "Volume")], names=["Price", "Ticker"])
frame = pd.DataFrame([[10.0, 10.1, 9.9, 10.0, 1], [11.0, 11.1, 10.9, 11.0, 2], [12.0, 12.1, 11.9, 12.0, 3]], index=days, columns=columns)

print("type of frame['Close']:", type(frame["Close"]).__name__)
wrong = pd.Series([1.0, 2.0, 3.0], index=days) - frame["Close"]
print("Series minus that DataFrame:", wrong.shape, "all NaN:", bool(wrong.isna().all().all()))
flat = data.normalize_download(frame)
print("after normalize_download:", type(flat["Close"]).__name__, flat["Close"].tolist())

## Module 3: cleaning and logs

The course fills gaps with `fillna(method='ffill')`. pandas 3.1 removed that keyword, so the call raises. `.ffill()` does the same job: it carries the last close forward. It only fills rows that exist; it never invents a trading day.

Then we take the natural log of the close. Forward filling and the log are *pointwise*: day t uses only day t and earlier. So they can run before the split without leaking the future into training.

In [ ]:
try:
    pd.Series([1.0, None, 3.0]).fillna(method="ffill")
except TypeError as error:
    print("course call raises TypeError:", error)
print(".ffill() gives:", pd.Series([1.0, None, 3.0]).ffill().tolist())
print("forward-filled dates in prices.json:", prices["fill"]["filled_dates"] or "none")

In [ ]:
# Optional: rebuild this notebook's payloads from the raw snapshot, if it is on this machine.
# (export.price_payloads fits no model; the whole export, with the AIC grid, takes minutes.)
try:
    snapshot = data.read_snapshot(config.raw_dir())
except data.SnapshotError:
    print("no raw snapshot under", config.raw_dir(), "- skipped (the cells below use the committed files)")
else:
    rebuilt, _ = export.price_payloads(snapshot)
    print("prices.json matches the snapshot:", rebuilt["prices"] == prices)
    print("eda.json matches the snapshot:   ", rebuilt["eda"] == eda_data)

## The split

The course window is split 90/10 **by position**, as course module 7 does it: the first `int(n * 0.9)` trading days train, the rest test. The out-of-sample window follows and is never used to choose anything.

In [ ]:
splits = prices["splits"]
for name in prep.SEGMENTS:
    s = splits[name]
    print(f"{name:>5}: {s['start']} to {s['end']}, {s['n']} rows")
course_n = splits["train"]["n"] + splits["test"]["n"]
print(f"int({course_n} * {splits['train_fraction']}) = {prep.train_size(course_n)}")

## Module 4: describing the training split only

The course describes the whole series. We describe the training rows only. What you notice about the test period can shape the model you pick, and then the test is no longer a test.

The next cell rebuilds `eda.json` from the training rows of `prices.json` and compares.

In [ ]:
table = table_from_prices(prices)
train = table[table["segment"] == "train"]
fresh = export.eda_payload(table)
print("eda.json matches a fresh computation:", fresh == eda_data)

summary = eda_data["summary"]
print("train rows:", summary["n"], "from", summary["start"], "to", summary["end"])
for key, value in summary["close_usd"].items():
    print(f"close {key:>6}: ${value:,.2f}")
for key, value in summary["log_return"].items():
    print(f"log return {key:>15}: {value:.6g}")

## KDEs and fat tails

A kernel density estimate puts a small normal bump on every value and adds them up. The bump's width is the bandwidth. Scott's rule, scipy's default, sets it to `n ** (-1/5)` times the sample standard deviation.

The course draws a KDE of the dollar close. A KDE of a price *level* mostly shows how long the price spent at each level, which is not a stable distribution. The log return is the series a model works with, and its KDE shows the fat tails: far more big days than a normal curve allows.

In [ ]:
for kde in eda_data["kdes"]:
    print(f"{kde['variable']:>10}: n = {kde['n']}, Scott factor {kde['bandwidth_factor']:.4f}, bandwidth {kde['bandwidth']:.6f}")
returns = eda.log_returns(train["log_close"])
tails = eda.tail_share(returns, 3.0)
print(f"returns beyond 3 sd: {tails['observed']:.2%} observed, {tails['normal']:.2%} under a normal")

## Decomposition: additive on logs, and the course's version

The course runs `seasonal_decompose(log_price, model='multiplicative', period=252)`. The log already turned "price = trend times season times noise" into "log price = trend + season + noise". Multiplying the logs again reads the log price as a product, which has no clear meaning.

We decompose the log additively. That is the multiplicative model in dollars: `exp(T + S + R) = e^T * e^S * e^R`. The cell measures how far the course's version moves from ours on the training rows.

In [ ]:
comparison = eda.course_comparison(train)
for key, value in comparison.items():
    print(f"{key:>28}: {value:.6g}")
rows = eda_data["decomposition"]["rows"]
head = next(i for i, r in enumerate(rows) if r["trend_log"] is not None)
print(f"trend undefined for the first {head} and last {sum(r['trend_log'] is None for r in rows) - head} rows")

## What comes next

Milestone 2 tests the training log price for a unit root (ADF and KPSS), picks d, and searches the (p, q) grid by AIC, all on the training rows printed above.